In [0]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Delta Assignment") \
    .getOrCreate()

In [0]:
# Step 1. Load dataset into Delta Table
df = spark.read.option("header", True).option("inferSchema",True).csv("/Volumes/azure_databricks/default/dataset/customer_master.csv")
display(df)

customer_id,name,city,email
101,Rahul Sharma,Delhi,rahul@gmail.com
102,Priya Verma,Mumbai,priya@gmail.com
103,Amit Patel,Ahmedabad,amit@gmail.com
104,Sneha Reddy,Hyderabad,sneha@gmail.com
105,Arjun Singh,Jaipur,arjun@gmail.com
106,Neha Gupta,Pune,neha@gmail.com
106,Neha Gupta,Pune,neha@gmail.com
107,null,Bhopal,abc@gmail.com


In [0]:
#Total Rows
print(df.count())

8


In [0]:
#Save as Delta Table
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("delta_table")
)

In [0]:
display(spark.table("delta_table"))

customer_id,name,city,email
101,Rahul Sharma,Delhi,rahul@gmail.com
102,Priya Verma,Mumbai,priya@gmail.com
103,Amit Patel,Ahmedabad,amit@gmail.com
104,Sneha Reddy,Hyderabad,sneha@gmail.com
105,Arjun Singh,Jaipur,arjun@gmail.com
106,Neha Gupta,Pune,neha@gmail.com
106,Neha Gupta,Pune,neha@gmail.com
107,null,Bhopal,abc@gmail.com


In [0]:
#Total Rows In Delta Table
print("Rows :", spark.table("delta_table").count())

Rows : 8


In [0]:
#STEP 2 Data Cleaning

#Remove NULL values
from pyspark.sql.functions import col
df_clean = spark.table("delta_table").dropna()

In [0]:
#Remove Duplicates
df_clean = df_clean.dropDuplicates()

In [0]:
# Cleaned data is saved in Delta Table
(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("delta_table")
)



In [0]:
#After removing NULL and Duplicate Values
print(df_clean.count())

6


In [0]:
#STEP 3  LOAD INCREMENTAL CSV AND READING IT
df_incremental = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/azure_databricks/default/dataset/customer_incremental.csv")
)

display(df_incremental)

customer_id,name,city,email
103,Amit Patel,Surat,amit@gmail.com
105,Arjun Singh,Udaipur,arjun@gmail.com
108,Karan Mehta,Indore,karan@gmail.com
109,Ananya Joshi,Nagpur,ananya@gmail.com


In [0]:
print(df_incremental.count())

4


In [0]:
#STEP 4 MERGE OPERATION 
from delta.tables import DeltaTable
delta_table = DeltaTable.forName(spark, "delta_table")

In [0]:
(
    delta_table.alias("target")
    .merge(
        df_incremental.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
).show()

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|                4|               2|               0|                2|
+-----------------+----------------+----------------+-----------------+



In [0]:
#STEP 5
#FINAL ROW COUNT
print("Final Row Count:", spark.table("delta_table").count())

Final Row Count: 8


In [0]:
#DUPLICATES CHECK AFTER CLEANING
from pyspark.sql.functions import count

dup_df = spark.table("delta_table").groupBy("customer_id").count().filter("count > 1")
dup_count = dup_df.count()

if dup_count == 0:
    print("No duplicate customer_id values found.")
else:
    print(f"{dup_count} duplicate customer_id value(s) found.")
    dup_df.show()

No duplicate customer_id values found.


In [0]:
#STEP 6  DISPLAY FINAL DATASET
display(spark.table("delta_table"))

customer_id,name,city,email
106,Neha Gupta,Pune,neha@gmail.com
102,Priya Verma,Mumbai,priya@gmail.com
101,Rahul Sharma,Delhi,rahul@gmail.com
104,Sneha Reddy,Hyderabad,sneha@gmail.com
103,Amit Patel,Surat,amit@gmail.com
105,Arjun Singh,Udaipur,arjun@gmail.com
108,Karan Mehta,Indore,karan@gmail.com
109,Ananya Joshi,Nagpur,ananya@gmail.com


In [0]:
# Week 7 Summary
#1. Load - Imported raw customer data from CSV and stored it as a Delta table.
#2. Clean - Removed nulls and duplicate rows, then saved the cleaned data back to the Delta table.
#3. Incremental Data - Loaded a second CSV to simulate new or updated customer records.
#4. Merge - Used Delta Lake's MERGE to update existing customers and add new ones based on customer_id.
#5. Validate - Checked row counts and duplicates to ensure the merge was successful and customer_ids are unique.
#6. Final Output: Displayed the updated Delta table.

#Conclusion: Delta Lake's MERGE provides robust incremental updates, handling both updates and inserts atomically.